# Strands + AgentCore Browser Tool - Live Viewer Implementation

This notebook demonstrates live browser automation with Strands AI analysis.

## Features
- **Live DCV Viewer**: Real-time browser visualization with AWS DCV streaming
- **Single Browser Session**: Shared session between live viewer and Playwright automation
- **Strands Analysis**: AI-powered content analysis with Bedrock Claude-3 Sonnet
- **Complete System**: Full analysis with screenshots, reports, and summaries
- **Universal Compatibility**: Works with any website and custom prompts

## Key Architecture
```
User Command → Browser Session → Live Viewer (DCV) 
                              → Playwright Automation (same session)
                              → Content Extraction
                              → Strands AI Analysis
                              → Complete Results Output
```

## 1. Environment Setup

In [ ]:
# Set up Python 3.12 virtual environment
!python3.12 --version
!python3.12 -m venv venv
!source venv/bin/activate && python --version

In [ ]:
# Install dependencies
!pip install --force-reinstall -U -r requirements.txt --quiet

print("✅ All dependencies installed successfully!")

## 2. Setup and Imports

In [ ]:
# Import required libraries
from bedrock_agentcore.tools.browser_client import browser_session
from strands import Agent
from strands.models import BedrockModel
from playwright.sync_api import sync_playwright
import sys
import os
import time
import webbrowser
import json
import base64
from datetime import datetime
from pathlib import Path

# Add interactive tools to path for BrowserViewerServer
interactive_tools_path = Path().absolute().parent / "interactive_tools"
sys.path.append(str(interactive_tools_path))

try:
    from browser_viewer import BrowserViewerServer
    print(f"✅ BrowserViewerServer imported from {interactive_tools_path}")
except ImportError as e:
    print(f"❌ BrowserViewerServer not found: {e}")
    BrowserViewerServer = None

print("✅ All libraries imported successfully!")

## 3. Complete Web Analysis System with Strands

In [ ]:
class WebAnalysisSystem:
    """Complete web analysis system with AgentCore, Playwright, and Strands"""
    
    def __init__(self, region="us-west-2", output_dir="analysis_results"):
        self.region = region
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Initialize Strands agent for analysis
        self.analysis_agent = self._create_analysis_agent()
        
    def _create_analysis_agent(self):
        """Create Strands agent for content analysis"""
        try:
            model = BedrockModel(model_id="anthropic.claude-3-sonnet-20240229-v1:0")
            
            agent = Agent(
                model=model,
                system_prompt="""You are an intelligent content analysis expert. You analyze any type of web content and provide relevant, comprehensive insights based on what the content actually contains.

**IMPORTANT**: Analyze the ACTUAL CONTENT and DATA on the page, not the webpage design or structure.

For different types of content, provide appropriate analysis:

**For Stock/Financial Pages:**
- Company overview and business model
- Financial performance metrics (revenue, earnings, ratios)
- Stock valuation analysis (P/E, market cap, price trends)
- Investment recommendation and risk assessment
- Market position and competitive analysis

**For News/Articles:**
- Key news points and their significance
- Impact analysis and implications
- Credibility and source assessment
- Related context and background

**For E-commerce/Products:**
- Product analysis and features
- Market positioning and pricing
- Customer value proposition
- Competitive landscape

**For General Websites:**
- Purpose and value proposition
- Key information and insights
- Target audience and use cases
- Business model and positioning

**Always Focus On:**
1. **Content Analysis**: What the actual data/information shows
2. **Key Insights**: Most important findings and takeaways
3. **Actionable Intelligence**: Practical insights for decision-making
4. **Context & Significance**: Why this information matters
5. **Recommendations**: Next steps or key considerations

Adapt your analysis style to match the content type. For financial data, be analytical and investment-focused. For news, be informative and contextual. For products, be evaluative and market-focused."""
            )
            
            print("✅ Analysis agent created")
            return agent
            
        except Exception as e:
            print(f"❌ Failed to create analysis agent: {e}")
            return None
    
    def analyze_website(self, url, analysis_name=None, custom_prompt=None):
        """Complete website analysis with screenshots and AI insights
        
        Args:
            url: Website URL to analyze
            analysis_name: Custom name for the analysis (optional)
            custom_prompt: Specific question or analysis request (optional)
        """
        
        if not analysis_name:
            analysis_name = f"analysis_{int(time.time())}"
        
        print(f"🔍 Starting complete analysis of: {url}")
        print(f"📁 Results will be saved to: {self.output_dir / analysis_name}")
        
        # Create analysis directory
        analysis_dir = self.output_dir / analysis_name
        analysis_dir.mkdir(exist_ok=True)
        
        results = {
            "url": url,
            "timestamp": datetime.now().isoformat(),
            "analysis_name": analysis_name,
            "screenshots": [],
            "text_content": "",
            "page_info": {},
            "ai_analysis": "",
            "success": False
        }
        
        try:
            # Step 1: Create browser session and capture data
            with browser_session(self.region) as client:
                print(f"✅ Browser session: {client.session_id}")
                
                # Start live viewer
                if BrowserViewerServer:
                    viewer = BrowserViewerServer(client, port=8000)
                viewer_url = viewer.start(open_browser=False)
                print(f"👀 Live viewer: {viewer_url}")
                
                # Open viewer for user to watch
                webbrowser.open(viewer_url)
                
                # Connect Playwright
                ws_url, headers = client.generate_ws_headers()
                
                with sync_playwright() as p:
                    browser = p.chromium.connect_over_cdp(ws_url, headers=headers)
                    context = browser.contexts[0] if browser.contexts else browser.new_context()
                    page = context.pages[0] if context.pages else context.new_page()
                    
                    print("✅ Playwright connected")
                    
                    # Navigate to target URL
                    print(f"🌐 Navigating to: {url}")
                    page.goto(url, wait_until="domcontentloaded", timeout=30000)
                    time.sleep(5)  # Wait for dynamic content
                    
                    # Capture page information
                    try:
                        results["page_info"] = {
                            "title": page.title(),
                            "url": page.url,
                            "viewport": page.viewport_size
                        }
                        print(f"📄 Page title: {results['page_info']['title']}")
                    except Exception as e:
                        print(f"⚠️  Could not get page info: {e}")
                    
                    # Capture text content
                    print("📄 Extracting text content...")
                    try:
                        text_content = page.inner_text('body')
                        results["text_content"] = text_content
                        
                        # Save text content to file
                        text_file = analysis_dir / "page_content.txt"
                        with open(text_file, 'w', encoding='utf-8') as f:
                            f.write(f"URL: {url}\n")
                            f.write(f"Title: {results['page_info'].get('title', 'N/A')}\n")
                            f.write(f"Captured: {results['timestamp']}\n")
                            f.write("=" * 80 + "\n\n")
                            f.write(text_content)
                        
                        print(f"✅ Text content saved: {text_file}")
                        
                    except Exception as e:
                        print(f"❌ Text extraction failed: {e}")
                    
                    # Take multiple screenshots
                    screenshot_types = [
                        ("full_page", {"full_page": True}),
                        ("viewport", {"full_page": False}),
                    ]
                    
                    for screenshot_name, options in screenshot_types:
                        try:
                            print(f"📸 Taking {screenshot_name} screenshot...")
                            screenshot_file = analysis_dir / f"{screenshot_name}.png"
                            page.screenshot(path=str(screenshot_file), **options)
                            
                            if screenshot_file.exists() and screenshot_file.stat().st_size > 1000:
                                results["screenshots"].append({
                                    "name": screenshot_name,
                                    "file": str(screenshot_file),
                                    "size": screenshot_file.stat().st_size
                                })
                                print(f"✅ Screenshot saved: {screenshot_file} ({screenshot_file.stat().st_size} bytes)")
                            else:
                                print(f"⚠️  Screenshot may be empty: {screenshot_file}")
                                
                        except Exception as e:
                            print(f"❌ Screenshot {screenshot_name} failed: {e}")
                    
                    # Wait a moment for user to see the live view
                    print(f"\n👀 Check the live viewer at: {viewer_url}")
                    print("⏱️  Waiting 10 seconds for you to observe the page...")
                    
                    for i in range(10, 0, -1):
                        print(f"   {i} seconds...", end='\r')
                        time.sleep(1)
                    
                    print("\n")
                    
            # Step 2: AI Analysis (outside browser session to avoid threading issues)
            if self.analysis_agent and results["text_content"]:
                print("🤖 Performing AI analysis...")
                
                # Build analysis prompt with custom request if provided
                base_prompt = f"""Analyze this web page content:

URL: {url}
Title: {results['page_info'].get('title', 'N/A')}
Captured: {results['timestamp']}

Page Content:
{results['text_content'][:8000]}  # Limit content to avoid token limits"""

                if custom_prompt:
                    analysis_prompt = f"""{base_prompt}

**SPECIFIC ANALYSIS REQUEST:**
{custom_prompt}

Focus your analysis on answering this specific request while providing relevant context from the page content."""
                else:
                    analysis_prompt = f"""{base_prompt}

Provide a comprehensive analysis following the guidelines in your system prompt."""

                try:
                    ai_analysis = self.analysis_agent(analysis_prompt)
                    results["ai_analysis"] = str(ai_analysis)
                    
                    # Save AI analysis
                    analysis_file = analysis_dir / "ai_analysis.md"
                    with open(analysis_file, 'w', encoding='utf-8') as f:
                        f.write(f"# AI Analysis Report\n\n")
                        f.write(f"**URL:** {url}\n")
                        f.write(f"**Title:** {results['page_info'].get('title', 'N/A')}\n")
                        f.write(f"**Analyzed:** {results['timestamp']}\n")
                        f.write(f"**Model:** Claude-3 Sonnet\n\n")
                        f.write("---\n\n")
                        f.write(str(ai_analysis))
                    
                    print(f"✅ AI analysis saved: {analysis_file}")
                    
                except Exception as e:
                    print(f"❌ AI analysis failed: {e}")
                    results["ai_analysis"] = f"Analysis failed: {str(e)}"
            
            # Step 3: Save complete results
            results["success"] = True
            results_file = analysis_dir / "analysis_results.json"
            
            with open(results_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            
            print(f"✅ Complete results saved: {results_file}")
            
            # Step 4: Generate summary report
            self._generate_summary_report(analysis_dir, results)
            
            return results
            
        except Exception as e:
            print(f"💥 Analysis failed: {e}")
            results["success"] = False
            results["error"] = str(e)
            return results
    
    def _generate_summary_report(self, analysis_dir, results):
        """Generate a comprehensive summary report"""
        
        report_file = analysis_dir / "SUMMARY_REPORT.md"
        
        with open(report_file, 'w', encoding='utf-8') as f:
            f.write(f"# Web Analysis Summary Report\n\n")
            f.write(f"**URL:** {results['url']}\n")
            f.write(f"**Analysis Date:** {results['timestamp']}\n")
            f.write(f"**Page Title:** {results['page_info'].get('title', 'N/A')}\n\n")
            
            f.write("## 📊 Analysis Overview\n\n")
            f.write(f"- **Success:** {'✅ Yes' if results['success'] else '❌ No'}\n")
            f.write(f"- **Screenshots Captured:** {len(results['screenshots'])}\n")
            f.write(f"- **Text Content Length:** {len(results['text_content'])} characters\n")
            f.write(f"- **AI Analysis:** {'✅ Completed' if results['ai_analysis'] and 'failed' not in results['ai_analysis'] else '❌ Failed'}\n\n")
            
            if results["screenshots"]:
                f.write("## 📸 Screenshots\n\n")
                for screenshot in results["screenshots"]:
                    f.write(f"- **{screenshot['name']}**: `{Path(screenshot['file']).name}` ({screenshot['size']:,} bytes)\n")
                f.write("\n")
            
            f.write("## 📄 Files Generated\n\n")
            f.write("- `page_content.txt` - Raw text content from the page\n")
            f.write("- `ai_analysis.md` - Comprehensive AI analysis\n")
            f.write("- `analysis_results.json` - Complete structured results\n")
            f.write("- `SUMMARY_REPORT.md` - This summary report\n")
            
            for screenshot in results["screenshots"]:
                f.write(f"- `{Path(screenshot['file']).name}` - {screenshot['name']} screenshot\n")
            
            f.write("\n## 🤖 AI Analysis Preview\n\n")
            if results["ai_analysis"] and "failed" not in results["ai_analysis"]:
                # Show first 500 characters of analysis
                preview = results["ai_analysis"][:500]
                f.write(f"{preview}...\n\n")
                f.write("*See `ai_analysis.md` for the complete analysis.*\n")
            else:
                f.write("AI analysis was not completed successfully.\n")
        
        print(f"✅ Summary report generated: {report_file}")

print("✅ WebAnalysisSystem class defined!")

## 4. Initialize the System

In [ ]:
# Get AWS region
import boto3
boto_session = boto3.Session()
region = boto_session.region_name or "us-west-2"

# Initialize the complete analysis system
system = WebAnalysisSystem(region=region, output_dir="analysis_results")

print("🚀 Complete Web Analysis System with Strands ready!")
print("\n📋 System capabilities:")
print("   ✅ Live DCV browser viewing")
print("   ✅ Single browser session (shared between viewer and automation)")
print("   ✅ Playwright automation on live session")
print("   ✅ Strands + Bedrock Claude-3 Sonnet")
print("   ✅ Complete analysis with screenshots and reports")
print("   ✅ Universal website compatibility")
print(f"\n🎯 Using AWS region: {region}")
print("\n🎬 Ready for complete website analysis!")

## 5. Usage Examples

Simple single-line commands to analyze any website with optional custom prompts.

### Example 1: News Analysis with Live Viewer

In [ ]:
# Analyze BBC News with live viewer - you'll see the browser automation in real-time!
result = system.analyze_website(
    url="https://www.bbc.com",
    analysis_name="bbc_news_analysis",
    custom_prompt="What are the top 3 news stories and their key points?"
)

if result["success"]:
    print("\n✅ Analysis completed successfully!")
    print(f"📁 Results saved to: analysis_results/{result['analysis_name']}")
    print(f"📸 Screenshots: {len(result['screenshots'])}")
    print(f"📄 Text content: {len(result['text_content'])} characters")
else:
    print(f"❌ Analysis failed: {result.get('error', 'Unknown error')}")

### Example 2: Stock Analysis

In [ ]:
# Analyze Tesla stock with live viewer
result = system.analyze_website(
    url="https://stockanalysis.com/stocks/tsla/",
    analysis_name="tesla_stock_analysis",
    custom_prompt="Extract Tesla's current stock price, market cap, and key financial metrics"
)

if result["success"]:
    print("\n✅ Stock analysis completed!")
    print(f"📊 Analysis saved to: analysis_results/{result['analysis_name']}")
else:
    print(f"❌ Analysis failed: {result.get('error', 'Unknown error')}")

### Example 3: E-commerce Product Analysis

### Example 4: GitHub Trending Analysis

In [ ]:
# Analyze GitHub trending repositories with live viewer
result = system.analyze_website(
    url="https://github.com/trending",
    analysis_name="github_trending_analysis",
    custom_prompt="What are the top 5 trending repositories and what technologies are they using?"
)

if result["success"]:
    print("\n✅ GitHub analysis completed!")
    print(f"🔥 Results saved to: analysis_results/{result['analysis_name']}")
else:
    print(f"❌ Analysis failed: {result.get('error', 'Unknown error')}")

## 6. View Analysis Results

In [ ]:
# List all analysis results
results_dir = Path("analysis_results")

print("📁 Analysis Results:")
if results_dir.exists():
    for analysis_folder in sorted(results_dir.iterdir()):
        if analysis_folder.is_dir():
            print(f"\n📂 {analysis_folder.name}/")
            for file in sorted(analysis_folder.glob("*")):
                size = file.stat().st_size
                print(f"   📄 {file.name} ({size:,} bytes)")
else:
    print("   No results yet - run an analysis first!")

In [ ]:
# Read and display a recent analysis result
import json

# Find the most recent analysis
results_dir = Path("analysis_results")
if results_dir.exists():
    analysis_folders = [f for f in results_dir.iterdir() if f.is_dir()]
    if analysis_folders:
        latest_folder = max(analysis_folders, key=lambda x: x.stat().st_mtime)
        
        # Read the AI analysis
        analysis_file = latest_folder / "ai_analysis.md"
        if analysis_file.exists():
            print(f"📊 Latest Analysis: {latest_folder.name}")
            print("=" * 50)
            
            with open(analysis_file, 'r', encoding='utf-8') as f:
                content = f.read()
                # Show first 1000 characters
                print(content[:1000] + "..." if len(content) > 1000 else content)
        
        # Show summary report
        summary_file = latest_folder / "SUMMARY_REPORT.md"
        if summary_file.exists():
            print("\n📋 Summary Report:")
            print("-" * 30)
            with open(summary_file, 'r', encoding='utf-8') as f:
                print(f.read())
    else:
        print("No analysis results found. Run an analysis first!")
else:
    print("No results directory found. Run an analysis first!")

## 7. System Summary

This notebook demonstrates a complete Strands + AgentCore browser integration with live viewing capabilities.

In [ ]:
print("🎉 Strands Complete Web Analysis System")
print("=" * 50)

print("\n✅ System Features:")
print("   🌐 Live DCV browser viewing with real-time automation")
print("   🤖 Strands AI analysis with Claude-3 Sonnet")
print("   🎭 Playwright automation on live session")
print("   📸 Screenshot capture (full page + viewport)")
print("   📁 Complete analysis reports with JSON, text, and markdown")
print("   � S-ummary reports with analysis overview")
print("   🔧 Command-line interface for automation")

print("\n🎯 Usage Patterns:")
print("   📈 Stock Analysis: Extract financial metrics and investment insights")
print("   📰 News Analysis: Extract headlines, summaries, and impact analysis")
print("   🛒 E-commerce: Product search, analysis, and competitive insights")
print("   � Diata Extraction: Specific information retrieval with context")
print("   🔍 Research: Market trends, technology analysis, and insights")

print("\n� Koey Advantages:")
print("   👀 Watch browser automation in real-time via live viewer")
print("   🎯 Custom analysis prompts for specific information needs")
print("   📋 Complete documentation with screenshots and reports")
print("   🌐 Universal compatibility with any website")
print("   🤖 Intelligent content analysis adapted to content type")

print("\n💡 The system is ready for production use!")
print("   Analyze any website with intelligent Strands AI insights.")
print("   Perfect for research, competitive analysis, and data extraction.")

print("\n🎯 Example Usage:")
print('   system.analyze_website("https://example.com", "my_analysis", "Extract key information")')